# Exploratory Data Analysis — Oil Markets and Big Mac Valuation Gaps

**IE 421 · Data Pilgrims · M2 Data Checkpoint**  
**Project:** When Oil Markets Shake, Do Burgers Tell a Different Story? A Big Mac Index Analysis

**Purpose.** Audit the joined data, inspect distributions and changes over time, then explore the three research questions without implying causal identification. All calculations use the canonical `data/merged_dataset.csv`; separate cleaned source datasets are used only for source-specific quality checks.

**Grain.** A merged row is one economy × one Big Mac publication edition. Brent movements repeat across economies in the same edition; **edition-level aggregation is essential when assessing Brent relationships**. Energy Institute oil classifications are approximations based on prior-year production minus consumption, not observed bilateral trade. Missing production has sometimes been treated as zero in the upstream pipeline; oil-group comparisons require caution.

## 1. Load data and verify the analysis environment

Run the notebook from either the repository root or `scripts/` in Jupyter/Colab after the project folder has been uploaded or mounted. The notebook does not silently download missing files or fabricate observations.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Use Okabe-Ito colours, with line styles and markers as redundant encodings.
BLUE = "#0072B2"
ORANGE = "#E69F00"
GREEN = "#009E73"
VERMILION = "#D55E00"
PURPLE = "#CC79A7"
INK = "#263345"
GREY = "#65717E"
GRID = "#E5E9EF"
plt.rcParams.update({"figure.dpi": 115, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})

# Resolve the project root from common Jupyter launch locations.
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/V1.3'), Path('/content')]
ROOT = next((candidate for candidate in candidates
             if (candidate / 'data' / 'merged_dataset.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError("Cannot locate data/merged_dataset.csv. Open the notebook from the extracted project root or scripts/ folder.")

panel = pd.read_csv(ROOT / 'data/merged_dataset.csv', parse_dates=['date'])
bigmac = pd.read_csv(ROOT / 'data/bigmac_clean.csv', parse_dates=['date'])
brent = pd.read_csv(ROOT / 'data/brent_daily.csv', parse_dates=['date'])
ei = pd.read_csv(ROOT / 'data/ei_oil_balance.csv')
tables = {'Merged analysis panel': panel, 'Big Mac': bigmac,
          'Brent': brent, 'Energy Institute': ei}
print('Project root:', ROOT)
print('Merged panel:', panel.shape)

### 1.1 First contact: types, sample, identifiers, missing values, duplicates

In [ ]:
keys = {'Merged analysis panel': ['iso_a3', 'date'],
        'Big Mac': ['iso_a3', 'date'], 'Brent': ['date'],
        'Energy Institute': ['iso_a3', 'year']}
quality_rows = []
for name, frame in tables.items():
    quality_rows.append({'dataset': name, 'rows': len(frame),
                         'columns': frame.shape[1], 'identifier': ' + '.join(keys[name]),
                         'duplicate_identifiers': int(frame.duplicated(keys[name]).sum()),
                         'missing_cells': int(frame.isna().sum().sum())})
quality = pd.DataFrame(quality_rows)
display(quality)
assert quality['duplicate_identifiers'].eq(0).all(), 'Duplicate primary keys found.'
print('Merged data types:')
display(panel.dtypes.rename('dtype').to_frame())
print('Reproducible random sample of merged rows:')
display(panel.sample(5, random_state=421))

In [ ]:
# Show missingness by variable as both counts and percentages.
missing = pd.DataFrame({'missing_rows': panel.isna().sum(),
                        'missing_pct': panel.isna().mean().mul(100)}).sort_values('missing_rows', ascending=False)
display(missing.round(2))
fig, ax = plt.subplots(figsize=(9, 4.2))
subset = missing[missing.missing_rows > 0].sort_values('missing_pct')
ax.barh(subset.index, subset.missing_pct, color=BLUE)
ax.set(xlabel='Missing observations (%)', title='Missingness in the merged analysis panel')
ax.set_xlim(0, max(subset.missing_pct.max() * 1.18, 1))
for idx, val in enumerate(subset.missing_pct):
    ax.text(val + 0.3, idx, f'{val:.1f}%', va='center', fontsize=9)
ax.grid(axis='x', color=GRID); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

In [ ]:
# Inspect why changes are missing: first observation versus gaps in the edition sequence.
panel_sorted = panel.sort_values(['iso_a3', 'date']).copy()
panel_sorted['first_for_economy'] = panel_sorted.groupby('iso_a3').cumcount().eq(0)
print('Economies:', panel.iso_a3.nunique())
print('Publication editions:', panel.date.nunique())
print('First observations with no previous economy edition:', int(panel_sorted.first_for_economy.sum()))
print('Missing oil role rows:', int(panel.oil_role.isna().sum()))
print('Missing change in Big Mac gap:', int(panel.USD_raw_chg.isna().sum()))
print('Missing Brent interval changes:', int(panel.brent_logchg_since_prev.isna().sum()))
print('Countries without oil classification:')
print(', '.join(sorted(panel.loc[panel.oil_role.isna(), 'name'].unique())))

**First-contact interpretation.** Missing changes are expected at the beginning of a time series and sometimes after a country skips an edition. Oil-role missingness reflects incomplete matching/coverage, not evidence that an economy has no oil exposure. No values are automatically imputed in this EDA.

## 2. One variable at a time

We inspect distribution, outliers, category counts, and time evolution before analysing relationships. `USD_raw` is a **ratio**, so multiplying by 100 expresses it as a percentage. The change in `USD_raw` is expressed in **percentage points** after multiplying by 100.

In [ ]:
continuous = ['USD_raw', 'USD_raw_chg', 'brent_avg_30d',
              'brent_logchg_since_prev', 'net_exports_kbd']
display(panel[continuous].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]).T.round(3))
print('Flagged extreme Big Mac gap rows:', int(panel.flag_extreme_raw.sum()))
print('Range of raw valuation gaps (%):', round(panel.USD_raw.min()*100, 1),
      'to', round(panel.USD_raw.max()*100, 1))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.hist(panel.USD_raw.dropna() * 100, bins=38, color=BLUE, edgecolor='white')
ax1.axvline(panel.USD_raw.median()*100, color=VERMILION, lw=2, ls='--',
            label=f'Median: {panel.USD_raw.median()*100:.1f}%')
ax1.set(xlabel='Big Mac valuation gap vs US (%)', ylabel='Economy-edition observations',
        title='Valuation gaps have an asymmetric distribution')
ax1.legend(frameon=False)
by_role = [panel.loc[panel.oil_role.eq(role), 'USD_raw_chg'].dropna() * 100
           for role in ('importer', 'exporter')]
box = ax2.boxplot(by_role, tick_labels=['Importer', 'Exporter'], showfliers=True,
                  patch_artist=True, widths=0.45)
for patch, color in zip(box['boxes'], [BLUE, ORANGE]):
    patch.set_facecolor(color); patch.set_alpha(0.65)
ax2.axhline(0, color=GREY, lw=1)
ax2.set(ylabel='Change in valuation gap (percentage points)',
        title='Changes vary within both oil-role groups')
for ax in (ax1, ax2):
    ax.grid(axis='y', color=GRID); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
counts = panel.groupby('date').iso_a3.nunique()
ax1.plot(counts.index, counts.values, color=BLUE, lw=2)
ax1.set(ylabel='Economies', xlabel='Publication date',
        title='Big Mac country coverage changes over editions')
# Use edition-level records, not repeated country rows, for oil-regime counts.
regimes = panel.dropna(subset=['oil_regime']).drop_duplicates('date').oil_regime.value_counts()
ax2.bar(['Calm', 'High-move'], [regimes.get('calm',0), regimes.get('high',0)],
        color=[GREEN, VERMILION])
ax2.set(ylabel='Publication intervals', title='Oil regimes: unique editions')
ax2.set_ylim(0, max(regimes.values)*1.2)
for ax in (ax1, ax2):
    ax.grid(axis='y', color=GRID); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print('Oil-role rows (not independent country counts):')
display(panel.oil_role.value_counts(dropna=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.7))
axes[0].plot(brent.date, brent.brent_usd, color=INK, lw=0.9)
axes[0].set(xlabel='Date', ylabel='USD per barrel',
            title='Daily Brent prices in the source series')
# Trading-day changes do not impute weekends or holidays.
positive = brent.loc[brent.brent_usd.gt(0)].sort_values('date').copy()
positive['daily_log_return_pct'] = np.log(positive.brent_usd).diff() * 100
axes[1].hist(positive.daily_log_return_pct.dropna(), bins=70, color=BLUE, edgecolor='white')
axes[1].set(xlabel='Daily log change (%)', ylabel='Trading-day observations',
            title='Brent daily changes: inspect extremes')
for ax in axes:
    ax.grid(color=GRID); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

**Distribution interpretation.** The source Brent series has much higher frequency than Big Mac publication editions. Apparent changes over time can reflect shifts in country coverage, different edition spacing, and market-wide events. No time-trend plot establishes that oil prices caused the Big Mac gap.

## 3. Relationships and research questions

For correlations between Brent changes and Big Mac changes, reduce the data to **one point per edition**. This avoids counting the same oil-price movement dozens of times as independent evidence.

In [ ]:
# Aggregate the country-specific gap changes within each publication edition.
changes = panel.dropna(subset=['USD_raw_chg', 'brent_logchg_since_prev', 'oil_regime']).copy()
edition = (changes.groupby(['date', 'oil_regime'], observed=True)
           .agg(mean_gap_change=('USD_raw_chg','mean'),
                median_gap_change=('USD_raw_chg','median'),
                n_economies=('iso_a3','nunique'),
                brent_log_change=('brent_logchg_since_prev','first'))
           .reset_index().sort_values('date'))
assert not edition.duplicated('date').any()
edition['brent_change_pct'] = 100 * np.expm1(edition.brent_log_change)
edition['mean_gap_change_pp'] = 100 * edition.mean_gap_change
edition['year'] = edition.date.dt.year
print(f'Edition-level observations for Brent relationships: {len(edition)}')
display(edition.head())

In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 5.0))
for role, color, marker, label in [('calm', GREEN, 'o', 'Calm'),
                                   ('high', VERMILION, 's', 'High-move')]:
    group = edition[edition.oil_regime.eq(role)]
    ax.scatter(group.brent_change_pct, group.mean_gap_change_pp,
               color=color, marker=marker, s=58, edgecolor='white',
               linewidth=0.7, label=f'{label} ({len(group)} editions)')
ax.axvline(0, color=GREY, lw=1); ax.axhline(0, color=GREY, lw=1)
ax.set(xlabel='Change in 30-day average Brent price since previous edition (%)',
       ylabel='Cross-country mean valuation-gap change (percentage points)',
       title='RQ1: Compare editions during calm and high-move intervals')
ax.legend(frameon=False); ax.grid(color=GRID); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
summary_rq1 = edition.groupby('oil_regime').agg(
    n_editions=('date','count'), median_gap_change_pp=('mean_gap_change_pp','median'),
    mean_abs_brent_move_pct=('brent_change_pct', lambda s: s.abs().mean()))
display(summary_rq1.round(2))
print('Overall edition-level Pearson correlation:',
      round(edition[['brent_change_pct','mean_gap_change_pp']].corr().iloc[0,1],3))

**RQ1 interpretation.** A scatterplot provides an exploratory relationship, not an intervention or causal estimate. Regimes are defined using the **magnitude** of standardized Brent change, not direction; avoid saying high-move automatically means rising oil prices. The fitted relationship and sampling uncertainty may differ by period.

In [ ]:
# Compare editions in four historical periods using the same axis scales.
def period_label(year):
    if year <= 2006: return '2001–2006'
    if year <= 2013: return '2007–2013'
    if year <= 2019: return '2014–2019'
    return '2020–2026'

edition['period'] = edition.year.map(period_label)
periods = ['2001–2006','2007–2013','2014–2019','2020–2026']
xlim = (edition.brent_change_pct.min()-8, edition.brent_change_pct.max()+8)
ylim = (edition.mean_gap_change_pp.min()-4, edition.mean_gap_change_pp.max()+4)
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True, sharey=True)
period_results = []
for ax, period in zip(axes.flat, periods):
    sub = edition[edition.period.eq(period)]
    r = sub[['brent_change_pct','mean_gap_change_pp']].corr().iloc[0,1]
    period_results.append({'period':period,'n_editions':len(sub),'pearson_r':r})
    for role, color, marker in [('calm',GREEN,'o'),('high',VERMILION,'s')]:
        d = sub[sub.oil_regime.eq(role)]
        ax.scatter(d.brent_change_pct, d.mean_gap_change_pp,
                   color=color, marker=marker, s=46, edgecolor='white')
    ax.axhline(0,color=GREY,lw=0.9); ax.axvline(0,color=GREY,lw=0.9)
    ax.set(xlim=xlim, ylim=ylim, title=f'{period} · n={len(sub)} · r={r:.2f}')
    ax.grid(color=GRID); ax.set_axisbelow(True)
for ax in axes[1]: ax.set_xlabel('Brent change (%)')
for ax in axes[:,0]: ax.set_ylabel('Mean valuation-gap change (pp)')
fig.suptitle('RQ2: The relationship may differ across historical periods',fontsize=13)
plt.tight_layout(); plt.show()
display(pd.DataFrame(period_results).round(3))

**RQ2 interpretation.** The period correlations have small edition sample sizes, and their values can be sensitive to a few unusual dates. They are descriptive diagnostics rather than formal proof of a regime change. A period-by-period slope comparison with uncertainty intervals belongs in a later modelling stage.

In [ ]:
# RQ3: use within-edition medians, rather than allowing larger editions to dominate.
role_data = panel.dropna(subset=['oil_role','USD_raw']).copy()
roles = (role_data.groupby(['date','oil_role'], observed=True)
         .agg(median_gap=('USD_raw','median'), n_economies=('iso_a3','nunique'))
         .reset_index())
fig, ax = plt.subplots(figsize=(10.5, 4.1))
for role, color, ls in [('exporter', ORANGE, '-'), ('importer', BLUE, '--')]:
    d = roles[roles.oil_role.eq(role)]
    ax.plot(d.date, d.median_gap*100, color=color, ls=ls, lw=2.3, label=role.title())
ax.axhline(0,color=GREY,lw=1)
ax.set(xlabel='Big Mac publication date', ylabel='Median valuation gap vs US (%)',
       title='RQ3: Exporter/importer valuation gaps over time')
ax.legend(frameon=False); ax.grid(color=GRID); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

# Eta-squared is a descriptive within-edition share of variation, not causal explanatory power.
def eta_squared(frame):
    frame = frame.dropna(subset=['USD_raw','oil_role'])
    if len(frame)<3 or frame.oil_role.nunique()<2: return np.nan
    mean = frame.USD_raw.mean()
    total = ((frame.USD_raw - mean)**2).sum()
    between = sum(len(part)*(part.USD_raw.mean()-mean)**2
                  for _,part in frame.groupby('oil_role'))
    return between/total if total>0 else np.nan

eta = role_data.groupby('date').apply(eta_squared, include_groups=False)
print('Median within-edition eta-squared:', round(eta.median(), 3))
print('Observed eta-squared range:', round(eta.min(),3), 'to', round(eta.max(),3))

**RQ3 interpretation.** The exporter/importer indicator is derived from a production-consumption balance, which is not an observed trade-flow measure. Missing or unlisted production in the source may misclassify countries. Differences in medians and eta-squared describe the available sample, not a causal role of being an exporter.

## 4. Sensitivity checks and limitations

Test whether descriptive results change when extremely large Big Mac gaps are excluded and when the aggregation uses **median** rather than mean. These are predeclared sensitivity checks; observations must not be removed only because they disagree with a desired conclusion.

In [ ]:
def edition_correlation(frame, agg='mean'):
    frame = frame.dropna(subset=['USD_raw_chg','brent_logchg_since_prev'])
    grouped = frame.groupby('date').agg(
        gap=('USD_raw_chg', agg), brent=('brent_logchg_since_prev','first'))
    if len(grouped)<3: return np.nan, len(grouped)
    r = grouped[['gap','brent']].corr().iloc[0,1]
    return r, len(grouped)

sensitivity = []
for label, frame in [('All rows', panel),
                     ('Exclude flagged extreme gaps', panel.loc[~panel.flag_extreme_raw])]:
    for method in ('mean','median'):
        r, n = edition_correlation(frame, method)
        sensitivity.append({'sample':label,'gap_aggregation':method,
                            'n_editions':n,'edition_level_pearson_r':r})
display(pd.DataFrame(sensitivity).round(3))

In [ ]:
# Check that the JSON export matches the merged CSV schema and row count.
json_path = ROOT / 'data/json/merged_dataset.json'
if json_path.exists():
    with json_path.open(encoding='utf-8') as f:
        records = json.load(f)
    assert isinstance(records, list)
    assert len(records) == len(panel)
    assert set(records[0]) == set(panel.columns)
    print(f'JSON parity: {len(records)} records, {len(records[0])} fields; matches CSV schema.')
else:
    print('JSON export not present; regenerate it with scripts/preprocess.py.')

print('Audit summary:')
print(f'  Merged records: {len(panel):,}; unique economies: {panel.iso_a3.nunique()}; editions: {panel.date.nunique()}')
print(f'  Edition-level relationship sample: {len(edition)}')
print(f'  Missing oil role: {panel.oil_role.isna().sum()} rows')
print(f'  Duplicate economy-date keys: {panel.duplicated(["iso_a3","date"]).sum()}')

## 5. Summary for the M2 checkpoint

- The analysis-ready dataset joins **The Economist Big Mac Index**, **EIA Brent prices**, and the **Energy Institute oil production/consumption extract**.
- The data span different sampling frequencies: edition-based, trading-day, and annual. The merged dataset follows **economy × edition** granularity.
- Missing oil classifications must stay missing; treating all unlisted producers as true zero-production economies is a limitation of the existing upstream transformation.
- RQ1–RQ2 exploration uses **one observation per edition** to avoid pseudo-replication from shared Brent values.
- RQ3 exporter/importer comparisons are descriptive and rely on an imperfect production-minus-consumption proxy.
- The included CSV/JSON snapshot can be analysed offline. **Full source regeneration is not validated** until the historical Energy Institute Excel workbook is provided and the download scripts are run with network access.

**Next:** Verify upstream regeneration, finalize data cards using actual raw/cleaned counts and source details, and publish a minimum of two figures with interpretations to the course website.